In [1]:
import os
import sys
from pathlib import Path

# Run this notebook from the repo root (audio-clf)
REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

os.environ["DATASETS_AUDIO_BACKEND"] = "soundfile"
os.environ["TORCHCODEC_QUIET"] = "1"

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

from load_data import load

dataset = load()

# Pick the held-out split the model never trained on
TEST_SPLIT = next(
    (s for s in ("test", "validation", "val") if s in dataset),
    list(dataset.keys())[-1]   # fallback: last split
)

print("Splits:", list(dataset.keys()))
for k in dataset.keys():
    print(f"  {k}: {len(dataset[k])} rows, columns: {dataset[k].column_names}")
print(f"\nUsing '{TEST_SPLIT}' as the held-out test split.")


Loading dataset...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Splits: ['train', 'validation', 'test']
  train: 9072 rows, columns: ['audio', 'text_description', 'transcribed_text', 'phonemes', 'duration', 'gender', 'age_category', 'emotion', 'lang', 'pitch', 'speech_monotony', 'speaking_rate', 'noise', 'sdr_noise', 'reverberation', 'category', 'source', 'speaker_id', 'original_name', 'start_time', 'end_time', 'augmented']
  validation: 504 rows, columns: ['audio', 'text_description', 'transcribed_text', 'phonemes', 'duration', 'gender', 'age_category', 'emotion', 'lang', 'pitch', 'speech_monotony', 'speaking_rate', 'noise', 'sdr_noise', 'reverberation', 'category', 'source', 'speaker_id', 'original_name', 'start_time', 'end_time', 'augmented']
  test: 504 rows, columns: ['audio', 'text_description', 'transcribed_text', 'phonemes', 'duration', 'gender', 'age_category', 'emotion', 'lang', 'pitch', 'speech_monotony', 'speaking_rate', 'noise', 'sdr_noise', 'reverberation', 'category', 'source', 'speaker_id', 'original_name', 'start_time', 'end_time',

In [2]:
from inference import resolve_checkpoint, load_model, run_row_inference
from load_data import get_row, read_audio
from train import SAMPLE_RATE
import numpy as np
from IPython.display import display, Audio

ckpt_path = resolve_checkpoint()
print(f"Using checkpoint: {ckpt_path}")

model, emotion_encoder, gender_encoder, age_encoder, \
    id2emotion, id2gender, id2age, processor = load_model(ckpt_path)
print("Model ready.")


Using checkpoint: /home/fatikh/issai/audio-clf/models/finetune/best_model_finetuned.pt
Model ready.


In [ ]:
def check_row(index: int, split: str | None = None):
    split = split or TEST_SPLIT
    row = get_row(split, index, dataset=dataset)

    audio_data, sample_rate = read_audio(row["audio"])
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    audio_data = audio_data.astype(np.float32)
    duration_sec = len(audio_data) / sample_rate
    audio_widget = Audio(data=audio_data, rate=sample_rate, normalize=True, embed=True)

    # ── Inference ──────────────────────────────────────────────────────────────
    results = run_row_inference(row, model, processor, id2emotion, id2gender, id2age)

    # ── Print summary ──────────────────────────────────────────────────────────
    sep = "─" * 56
    print(sep)
    print(f"  Row {index}  |  split: '{split}'  |  {duration_sec:.2f}s")
    transcript = row.get("transcribed_text") or row.get("text") or row.get("transcript") or "—"
    description = row.get("text_description", "")
    print(f"  Transcript : {transcript}")
    if description:
        print(f"  Description: {description}")
    print(sep)
    for task, gt_key in [("emotion", "emotion"), ("gender", "gender"), ("age", "age_category")]:
        r = results[task]
        gt = row.get(gt_key, "—")
        match = "✓" if r["pred"] == str(gt) else "✗"
        print(f"\n  {task.capitalize():<10} pred: {r['pred']} ({r['confidence']*100:.1f}%)  {match}"
              f"   gt: {gt}")
        print(f"             top-3: {[(lbl, f'{p*100:.1f}%') for lbl, p in r['top3']]}")
    print(f"\n{sep}")

    display(audio_widget)
    return audio_widget, results


In [5]:
# ── Change index to inspect any row from the test split ──
audio, results = check_row(index=5)

────────────────────────────────────────────────────────
  Row 5  |  split: 'test'  |  15.76s
  Transcript : Отыра беріңіз, қоя беріңіз, мен барлығына дайынмын деп біздің Артур кәдімгідей дайын отыр, достар. Иә, бастайық. Артур, енді сен ән әлеміне қалай келдің, немесе кім әсер етті, әлде бала кезден музыкаға қызығушылығың болды ма?
  Description: A female adult speaker with a high pitch, very expressive and happy tone, speaking at a low pace. The audio is clear with low noise and no reverberation.
────────────────────────────────────────────────────────

  Emotion    pred: happy (89.9%)  ✓   gt: happy
             top-3: [('happy', '89.9%'), ('neutral', '8.5%'), ('angry', '1.5%')]

  Gender     pred: F (100.0%)  ✓   gt: F
             top-3: [('F', '100.0%'), ('M', '0.0%')]

  Age        pred: adult (69.8%)  ✓   gt: adult
             top-3: [('adult', '69.8%'), ('young', '30.2%'), ('child', '0.0%')]

────────────────────────────────────────────────────────
